In [7]:
import pandas as pd
import numpy as np

data = {
    'id_pracownika': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,3,7,12,18,5,21,22,23,24,25],
    'imie': ['Anna', 'Bartek', 'CELINA', 'darek', 'Ewa', 'Filip', 'Gosia', 'HENRYK', 'irena', 'Jan', 'Kasia', 'Leszek', 'Marta', 'norbert', 'OLGA', 'Piotr', 'Renata', 'sławek', 'Teresa', 'Urszula', 'CELINA', 'Gosia', 'Leszek', 'sławek', 'Ewa', 'Wanda', 'Xawery', 'Yvonne', 'Zbyszek', 'Agata'],
    'dzial': ['Sprzedaz', 'IT', 'HR', 'sprzedaz', 'IT', 'HR', 'Sprzedaz', 'it', 'HR', 'Sprzedaz', 'IT', 'HR', 'sprzedaz', 'IT', 'HR', 'Sprzedaz', 'it', 'HR', 'Sprzedaz', 'IT', 'HR', 'Sprzedaz', 'HR', 'HR', 'IT', 'Sprzedaz', 'IT', 'hr', 'Sprzedaz', 'IT'],
    'wynagrodzenie': ['4500', '6200', '3800', '5100', '7500', '4200', '5800', '6900', '3500', '4800', '7200', '4100', '5500', '6800', '3900', '5200', '4700', '6100', '3800', '5400', '3800', '5800', '4100', '6100', '7500', 'brak', '5000', None, '4300', '6600'],
    'data_zatrudnienia': ['2020-03-15', '2019-07-22', '2021-01-10', '2018-05-30', '2022-11-01', '2020-08-14', '2019-12-05', '2021-06-18', '2017-09-23', '2023-02-07', '2020-04-11', '2018-11-28', '2022-07-15', '2019-03-19', '2021-10-08', '2020-06-25', '2018-08-31', '2022-02-14', '2019-05-07', '2021-09-20', '2021-01-10', '2019-12-05', '2018-11-28', '2022-02-14', '2022-11-01', '2023-01-15', '2022-05-20', None, '2020-10-11', '2019-08-03'],
    'ocena_roczna': [4.5, 3.8, None, 4.2, 5.0, 3.5, 4.7, None, 4.1, 3.9, 5.0, 4.3, 3.6, None, 4.8, 3.7, 4.4, 4.9, None, 3.8, None, 4.7, 4.3, 4.9, 5.0, 4.6, None, 4.9, 3.5, 4.1]
}

df = pd.DataFrame(data)

In [8]:
# 1a Diagnoza
print(df.info())
print("\nBraki w kolumnach:\n", df.isna().sum())

# 1b Naprawa wynagrodzenia
df['wynagrodzenie'] = df['wynagrodzenie'].replace('brak', np.nan)
df['wynagrodzenie'] = pd.to_numeric(df['wynagrodzenie'], errors='coerce')

# 1c Strategie fillna
mediana_wyn = df['wynagrodzenie'].median()
df['wynagrodzenie'] = df['wynagrodzenie'].fillna(mediana_wyn)

srednia_ocena = round(df['ocena_roczna'].mean(), 2)
df['ocena_roczna'] = df['ocena_roczna'].fillna(srednia_ocena)

print(f"Uzupełniono: Mediana wynagrodzeń={mediana_wyn}, Średnia ocen={srednia_ocena}")

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_pracownika      30 non-null     int64  
 1   imie               30 non-null     str    
 2   dzial              30 non-null     str    
 3   wynagrodzenie      29 non-null     str    
 4   data_zatrudnienia  29 non-null     str    
 5   ocena_roczna       24 non-null     float64
dtypes: float64(1), int64(1), str(4)
memory usage: 1.5 KB
None

Braki w kolumnach:
 id_pracownika        0
imie                 0
dzial                0
wynagrodzenie        1
data_zatrudnienia    1
ocena_roczna         6
dtype: int64
Uzupełniono: Mediana wynagrodzeń=5150.0, Średnia ocen=4.34


In [9]:
# 2a i b Usunięcie duplikatów
print(f"Liczba pełnych duplikatów: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

# 2c Konwersja dat
df['data_zatrudnienia'] = pd.to_datetime(df['data_zatrudnienia'], errors='coerce')
df['rok_zatrudnienia'] = df['data_zatrudnienia'].dt.year

print(f"Nowy kształt danych: {df.shape}")

Liczba pełnych duplikatów: 5
Nowy kształt danych: (25, 7)


In [10]:
#czyszczenie(Imiona i Działy)
df['imie'] = df['imie'].str.strip().str.title()
df['dzial'] = df['dzial'].str.strip().str.title()
df['dzial'] = df['dzial'].replace({'Hr': 'HR', 'It': 'IT'})

#Raport
print("\n=== RAPORT HR ===")
raport = df.groupby('dzial').agg({
    'wynagrodzenie': 'mean',
    'ocena_roczna': 'mean',
    'id_pracownika': 'count'
}).round(2).rename(columns={'id_pracownika': 'liczba_osob'})

print(raport)

print("\nTOP 3 najlepiej zarabiających:")
print(df.nlargest(3, 'wynagrodzenie')[['imie', 'dzial', 'wynagrodzenie']])


=== RAPORT HR ===
          wynagrodzenie  ocena_roczna  liczba_osob
dzial                                             
HR              4392.86          4.41            7
IT              6255.56          4.35            9
Sprzedaz        4905.56          4.12            9

TOP 3 najlepiej zarabiających:
      imie dzial  wynagrodzenie
4      Ewa    IT         7500.0
10   Kasia    IT         7200.0
7   Henryk    IT         6900.0


In [11]:
# 4a i b Analiza tekstu
imiona_na_a = df[df['imie'].str.startswith('A')]
print(f"Liczba osób z imieniem na 'A': {len(imiona_na_a)}")

df['dlugosc_imienia'] = df['imie'].str.len()

# 4c
it_hr = df[df['dzial'].str.contains('IT|HR')]
print(f"Pracownicy w działach wspierających (IT/HR): {len(it_hr)}")

Liczba osób z imieniem na 'A': 2
Pracownicy w działach wspierających (IT/HR): 16
